# FuNOVA Screen — Spec-Driven UMAP runs

Write *what data to show* and *how to color it* as plain Python dicts, then run.
The helper `submit_runs()` (in `manuscript/funova_screen_run_helpers.py`)
generates matching `FigureConfig` + `PlotConfig` classes, writes them to two
`_generated.py` files alongside the helper, and prints the bsub commands.

**Defaults** come from `FuNOVA_Screen_BaseFigureConfig` /
`FuNOVA_Screen_BasePlotConfig` (in `manuscript/`):
- `EXPERIMENT_TYPE='FuNOVA_Screen'`, `INPUT_FOLDERS=['batch1']`, `CELL_LINES=['C9']`
- `MARKERS_TO_EXCLUDE=['HDGFL2','FK-2']`
- 11 marker colors, 2 rep colors, ~192 auto-generated condition colors

**Spec keys you can override per run:**

| `data` key | `plot` key |
|---|---|
| `name` (required) | `color_by` (required: `rep`/`batch`/`condition`/`cell_line`/`cell_line_condition`/`marker`) |
| `input_folders` | `umap_type` (default chosen from `color_by`) |
| `cell_lines`, `conditions` | `size`, `alpha`, `figsize` |
| `markers`, `markers_to_exclude` | `color_overrides` ({item: '#hex'}) — patches auto colors |
| `add_rep_to_label`, `add_batch_to_label` | `color_mappings` — fully replaces COLOR_MAPPINGS |
| `show_ari`, `saveroot_infix`, `experiment_type` | |

`submit_runs(runs, memory=10000, dir='$run_type', submit=False)` — flip `submit=True` to actually launch bsubs.

In [1]:
import os, sys

NOVA_HOME =  "/home/projects/hornsteinlab/giliwo/NOVA" 
os.environ["NOVA_HOME"] = NOVA_HOME
sys.path.insert(0, NOVA_HOME)
sys.path.insert(0, os.path.dirname(NOVA_HOME))  # so `NOVA.manuscript.X` also resolves
print("NOVA_HOME:", NOVA_HOME)

# Imports — helper, plate condition lists, defaults
from manuscript.funova_screen_run_helpers import submit_runs
from manuscript.FuNOVA_Screen_Conditions_Lists import (
    plate1_conditions, plate2_conditions, plate3_conditions, plate4_conditions,    plate1_control_conditions, plate2_control_conditions, plate3_control_conditions, plate4_control_conditions,
    plate1_kds, plate2_kds, plate3_kds, plate4_kds,
)
from manuscript.manuscript_figures_data_config_FuNOVA_Screen import (
    FUNOVA_SCREEN_ALL_CONDITIONS, FUNOVA_SCREEN_CONDITIONS_PLATES,FUNOVA_SCREEN_CONTROL_CONDITIONS_PLATES, FUNOVA_SCREEN_KDS_CONDITIONS_PLATES
)
print("Plates:", [(k, len(v)) for k, v in FUNOVA_SCREEN_CONDITIONS_PLATES.items()])
print("Total conditions:", len(FUNOVA_SCREEN_ALL_CONDITIONS))

NOVA_HOME: /home/projects/hornsteinlab/giliwo/NOVA
Plates: [('plate1', 48), ('plate2', 48), ('plate3', 48), ('plate4', 48)]
Total conditions: 192


In [2]:
# One-time setup. NOVA_HOME must point to the giliwo NOVA clone for the helpers' imports.
import os, sys

NOVA_HOME =  "/home/projects/hornsteinlab/Collaboration/NOVA" #os.environ.get("NOVA_HOME", "home/projects/hornsteinlab/Collaboration/NOVA")
MODEL_PATH = "/home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen"
OUTDIR_NAME = "Funova_Screen_UMAPS_logs"
os.environ["NOVA_HOME"] = NOVA_HOME
os.environ["MODEL_PATH"] = MODEL_PATH
print(os.getcwd())
sys.path.insert(0, os.getcwd())
sys.path.insert(1, NOVA_HOME)


print("NOVA_HOME:", NOVA_HOME)
print("Python:", sys.executable)

/home/projects/hornsteinlab/giliwo
NOVA_HOME: /home/projects/hornsteinlab/Collaboration/NOVA
Python: /home/projects/hornsteinlab/giliwo/.conda/envs/nova/bin/python


## Filter out "Empty" wells

Re-binds the plate dicts and `FUNOVA_SCREEN_ALL_CONDITIONS` to versions that drop
every `Empty-*` condition. After running this cell, all run-spec cells below
(Examples 1, 5, 6 …) will automatically exclude the empty wells, since they all
read these names. Set `EXCLUDE_EMPTY = False` to keep them.

In [3]:
EXCLUDE_EMPTY = True

def _drop_empty(conds):
    return [c for c in conds if not c.startswith("Empty")]

if EXCLUDE_EMPTY:
    FUNOVA_SCREEN_CONDITIONS_PLATES         = {p: _drop_empty(v) for p, v in FUNOVA_SCREEN_CONDITIONS_PLATES.items()}
    FUNOVA_SCREEN_CONTROL_CONDITIONS_PLATES = {p: _drop_empty(v) for p, v in FUNOVA_SCREEN_CONTROL_CONDITIONS_PLATES.items()}
    FUNOVA_SCREEN_KDS_CONDITIONS_PLATES     = {p: _drop_empty(v) for p, v in FUNOVA_SCREEN_KDS_CONDITIONS_PLATES.items()}
    FUNOVA_SCREEN_ALL_CONDITIONS            = _drop_empty(FUNOVA_SCREEN_ALL_CONDITIONS)
    print("Dropped 'Empty-*' conditions.")

print("Plates after filter:", [(k, len(v)) for k, v in FUNOVA_SCREEN_CONDITIONS_PLATES.items()])
print("Total conditions:   ", len(FUNOVA_SCREEN_ALL_CONDITIONS))

Dropped 'Empty-*' conditions.
Plates after filter: [('plate1', 47), ('plate2', 47), ('plate3', 47), ('plate4', 47)]
Total conditions:    188


## Example 1 — UMAP0 by condition, one figure per plate

Most UMAPs separated into plates (each plate has ~50 conditions).

all conditions:

In [4]:
runs_per_plate = [
    {"data": {"name": f"{plate_label}_AllCond", "conditions": conds},
     "plot": {"color_by": "condition", "umap_type": 0, "size": 3}}
    for plate_label, conds in FUNOVA_SCREEN_CONDITIONS_PLATES.items()
]
submit_runs(runs_per_plate, memory=10000, dir=f"./{OUTDIR_NAME}/funova_screen_umap0_per_plate", submit=False, nova_home=NOVA_HOME, model_path=MODEL_PATH);

Data: +0 new, 4 already present (file now has 156 classes)
Plot: +0 new, 4 already present (file now has 156 classes)
  data classes kept (not regenerated): ['FuNOVA_Screen_Data_plate1_AllCond', 'FuNOVA_Screen_Data_plate2_AllCond', 'FuNOVA_Screen_Data_plate3_AllCond', 'FuNOVA_Screen_Data_plate4_AllCond']
  plot classes kept (not regenerated): ['FuNOVA_Screen_Plot_plate1_AllCond_condition', 'FuNOVA_Screen_Plot_plate2_AllCond_condition', 'FuNOVA_Screen_Plot_plate3_AllCond_condition', 'FuNOVA_Screen_Plot_plate4_AllCond_condition']
"bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_per_plate/FuNOVA_Screen_Data_plate1_AllCond_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_plate1_AllCond

only controls:

In [5]:
runs_per_plate = [
    {"data": {"name": f"{plate_label}_ControlCond", "conditions": conds},
     "plot": {"color_by": "condition", "umap_type": 0}}
    for plate_label, conds in FUNOVA_SCREEN_CONTROL_CONDITIONS_PLATES.items()
]
submit_runs(runs_per_plate, memory=10000, dir=f"./{OUTDIR_NAME}/funova_screen_umap0_per_plate", submit=False, nova_home=NOVA_HOME, model_path=MODEL_PATH);

Data: +0 new, 4 already present (file now has 156 classes)
Plot: +0 new, 4 already present (file now has 156 classes)
  data classes kept (not regenerated): ['FuNOVA_Screen_Data_plate1_ControlCond', 'FuNOVA_Screen_Data_plate2_ControlCond', 'FuNOVA_Screen_Data_plate3_ControlCond', 'FuNOVA_Screen_Data_plate4_ControlCond']
  plot classes kept (not regenerated): ['FuNOVA_Screen_Plot_plate1_ControlCond_condition', 'FuNOVA_Screen_Plot_plate2_ControlCond_condition', 'FuNOVA_Screen_Plot_plate3_ControlCond_condition', 'FuNOVA_Screen_Plot_plate4_ControlCond_condition']
"bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_per_plate/FuNOVA_Screen_Data_plate1_ControlCond_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generat

## Example 2 — UMAP1 (multi-marker), all conditions, with and without DAPI

In [6]:
runs_umap1 = [
    {"data": {"name": "AllCond_AllMarkers"},
     "plot": {"color_by": "marker", "umap_type": 1}},
    {"data": {"name": "AllCond_woDAPI",
              "markers_to_exclude": ["HDGFL2", "FK-2", "DAPI"]},
     "plot": {"color_by": "marker", "umap_type": 1}},
]
submit_runs(runs_umap1, memory=20000, dir=f"./{OUTDIR_NAME}/funova_screen_umap1", submit=False);

Data: +0 new, 2 already present (file now has 156 classes)
Plot: +0 new, 2 already present (file now has 156 classes)
  data classes kept (not regenerated): ['FuNOVA_Screen_Data_AllCond_AllMarkers', 'FuNOVA_Screen_Data_AllCond_woDAPI']
  plot classes kept (not regenerated): ['FuNOVA_Screen_Plot_AllCond_AllMarkers_marker', 'FuNOVA_Screen_Plot_AllCond_woDAPI_marker']
"bsub -q short -R rusage[mem=20000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap1/FuNOVA_Screen_Data_AllCond_AllMarkers_0.out -J umap_0 python None/runnables/generate_umaps_and_plot.py $MODEL_PATH ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_AllCond_AllMarkers ./NOVA/manuscript/manuscript_plot_config_FuNOVA_Screen_generated/FuNOVA_Screen_Plot_AllCond_AllMarkers_marker"
"bsub -q short -R rusage[mem=20000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap1/FuNOVA_Screen_Data_AllCond_woDAPI_1.out -J umap_1 python None/runnables/generate_umaps_and_plot.py $MODEL_PATH ./NOVA/manus

## Example 3 — QC: UMAP0 by reps and by batches

In [7]:
runs_qc = [
    {"data": {"name": "AllData_ByReps"},     "plot": {"color_by": "rep"}},
    {"data": {"name": "AllData_ByBatches"},  "plot": {"color_by": "batch"}},
]
submit_runs(runs_qc, memory=24000, dir=f"./{OUTDIR_NAME}/funova_screen_umap0_per_reps_and_batches", submit=False, nova_home=NOVA_HOME, model_path=MODEL_PATH);

Data: +0 new, 2 already present (file now has 156 classes)
Plot: +0 new, 2 already present (file now has 156 classes)
  data classes kept (not regenerated): ['FuNOVA_Screen_Data_AllData_ByReps', 'FuNOVA_Screen_Data_AllData_ByBatches']
  plot classes kept (not regenerated): ['FuNOVA_Screen_Plot_AllData_ByReps_rep', 'FuNOVA_Screen_Plot_AllData_ByBatches_batch']
"bsub -q short -R rusage[mem=24000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_per_reps_and_batches/FuNOVA_Screen_Data_AllData_ByReps_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_AllData_ByReps ./NOVA/manuscript/manuscript_plot_config_FuNOVA_Screen_generated/FuNOVA_Screen_Plot_AllData_ByReps_rep"
"bsub -q short -R rusage[mem=24000] -o ./Funova_Screen_U

## Example 4 — Highlight a few hits per plate (color overrides)

Color most conditions gray, highlight a chosen list. `color_overrides` patches the auto palette.

In [8]:
hits_to_highlight = ["TDP-43_p3", "Ranbp17_p3", "HMGCS1", "PIK3C3", "MAPKAP1"]
gray = "#CCCCCC"

# build {cond: gray} for every condition, then highlight the hits in red
overrides = {c: gray for c in FUNOVA_SCREEN_ALL_CONDITIONS}
for hit in hits_to_highlight:
    if hit in overrides:
        overrides[hit] = "#E41A1C"

runs_highlight = [
    {"data": {"name": "AllCond_HighlightHits"},
     "plot": {"color_by": "condition", "umap_type": 0,
              "color_overrides": overrides}},
]
submit_runs(runs_highlight, memory=10000, dir="funova_screen_highlight", submit=False);

Data: +0 new, 1 already present (file now has 156 classes)
Plot: +0 new, 1 already present (file now has 156 classes)
  data classes kept (not regenerated): ['FuNOVA_Screen_Data_AllCond_HighlightHits']
  plot classes kept (not regenerated): ['FuNOVA_Screen_Plot_AllCond_HighlightHits_condition']
"bsub -q short -R rusage[mem=10000] -o funova_screen_highlight/FuNOVA_Screen_Data_AllCond_HighlightHits_0.out -J umap_0 python None/runnables/generate_umaps_and_plot.py $MODEL_PATH ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_AllCond_HighlightHits ./NOVA/manuscript/manuscript_plot_config_FuNOVA_Screen_generated/FuNOVA_Screen_Plot_AllCond_HighlightHits_condition"


## Example 5 — Binary coloring: control vs. KD

Same per-plate UMAPs as Example 1, but every control well is one colour and every
KD well is another, regardless of the specific gene. Useful for a quick visual
read on whether KDs separate from controls.

`color_overrides` is built per plate from `FUNOVA_SCREEN_CONTROL_CONDITIONS_PLATES` —
any condition listed there gets `CTRL_COLOR`, anything else (KD) gets `KD_COLOR`.

In [9]:
for plate_label, conds in FUNOVA_SCREEN_CONDITIONS_PLATES.items():
    print(plate_label, len(conds))

plate1 47
plate2 47
plate3 47
plate4 47


In [10]:
# Binary control / KD colouring, one UMAP per plate.
CTRL_COLOR = "#1F77B4"   # blue  → controls (incl. Empty + non-targeting)
KD_COLOR   = "#D62728"   # red   → all KDs

runs_binary = []
for plate_label, conds in FUNOVA_SCREEN_CONDITIONS_PLATES.items():
    controls = set(FUNOVA_SCREEN_CONTROL_CONDITIONS_PLATES[plate_label])
    overrides = {c: (CTRL_COLOR if c in controls else KD_COLOR) for c in conds}
    runs_binary.append({
        "data": {"name": f"{plate_label}_BinaryCtrlKD", "conditions": conds},
        "plot": {"color_by": "condition", "umap_type": 0, "size": 30,
                 "color_overrides": overrides},
    })

submit_runs(
    runs_binary, memory=10000,
    dir=f"./{OUTDIR_NAME}/funova_screen_umap0_binary_ctrl_kd",
    submit=False, nova_home=NOVA_HOME, model_path=MODEL_PATH,
);

Data: +0 new, 4 already present (file now has 156 classes)
Plot: +0 new, 4 already present (file now has 156 classes)
  data classes kept (not regenerated): ['FuNOVA_Screen_Data_plate1_BinaryCtrlKD', 'FuNOVA_Screen_Data_plate2_BinaryCtrlKD', 'FuNOVA_Screen_Data_plate3_BinaryCtrlKD', 'FuNOVA_Screen_Data_plate4_BinaryCtrlKD']
  plot classes kept (not regenerated): ['FuNOVA_Screen_Plot_plate1_BinaryCtrlKD_condition', 'FuNOVA_Screen_Plot_plate2_BinaryCtrlKD_condition', 'FuNOVA_Screen_Plot_plate3_BinaryCtrlKD_condition', 'FuNOVA_Screen_Plot_plate4_BinaryCtrlKD_condition']
"bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_binary_ctrl_kd/FuNOVA_Screen_Data_plate1_BinaryCtrlKD_0.out -J umap_0 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_

## Example 6 — Each KD vs. each control (distinct UMAPs, per plate)

For every plate, build a UMAP per `(kd, control)` pair that contains only those
two conditions. Each UMAP is independent (own embedding, own figure). KD is
always red and the chosen control always blue, so plots are visually comparable
across pairs.

⚠️ **High volume.** This produces `n_plates × n_kds × n_controls` runs (≈ 1000 if
you keep every plate). Inspect the printed count first, narrow `PLATES_TO_RUN` /
`KDS_FILTER` / `CONTROLS_FILTER` as needed, and only flip `submit=True` once
you're happy with the list.

In [16]:
plate1_tdp =  ["CTNNB1", "GNB1L", "FNBP1L","LDLR", "GAK", "FOXK1"] # V
plate2_tdp = ["TELO2", "TBCE", "SYT16", "TMEM50B","POLR3E","NUP133"]
plate3_tdp = ["CSNK1E", "MYCN","NADSYN1","POLI","GNAL","DCX"]
plate4_tdp = ["TDP1", "HMGCS1","BMP2K","RCOR2","CACNA2D2","KIAA0232"]
# Each KD vs. each control — distinct UMAPs, per plate.
# Narrow the scope by editing the filters below; leave them as-is to run everything.
PLATES_TO_RUN    = ["plate3"] #list(FUNOVA_SCREEN_KDS_CONDITIONS_PLATES.keys())  # e.g. ["plate4"] for one plate
KDS_FILTER       = plate3_tdp        # None = all KDs in the plate; or a list like ["AKIRIN2", "TDP-43-p4"]
CONTROLS_FILTER  = None        # None = all controls in the plate; or a list of control names

# Slice the KD list per plate AFTER the filter, so you can submit one chunk at a time.
# Examples:
#   KD_SLICE = slice(None)     # all KDs (default)
#   KD_SLICE = slice(0, 10)    # first 10 KDs
#   KD_SLICE = slice(10, 20)   # KDs 10..19
#   KD_SLICE = slice(40, None) # KDs 40..end
KD_SLICE = slice(3,6)

CTRL_COLOR = "#1F77B4"
KD_COLOR   = "#D62728"

def _slice_tag(s: slice) -> str:
    start = 0 if s.start is None else s.start
    stop  = "end" if s.stop is None else s.stop
    return f"{start}_{stop}"

slice_tag = _slice_tag(KD_SLICE)

runs_kd_vs_ctrl = []
for plate_label in PLATES_TO_RUN:
    kds   = list(FUNOVA_SCREEN_KDS_CONDITIONS_PLATES[plate_label])
    ctrls = list(FUNOVA_SCREEN_CONTROL_CONDITIONS_PLATES[plate_label])
    if KDS_FILTER      is not None: kds   = [k for k in kds   if k in KDS_FILTER]
    if CONTROLS_FILTER is not None: ctrls = [c for c in ctrls if c in CONTROLS_FILTER]
    kds = kds[KD_SLICE]
    if not kds:
        print(f"  skipping {plate_label}: no KDs in slice {KD_SLICE}")
        continue

    for kd in kds:
        for ctrl in ctrls:
            runs_kd_vs_ctrl.append({
                "data": {
                    "name": f"{plate_label}_{kd}_vs_{ctrl}",
                    "conditions": [kd, ctrl],
                },
                "plot": {
                    "color_by": "condition", "umap_type": 0, "size": 30,
                    "color_overrides": {kd: KD_COLOR, ctrl: CTRL_COLOR},
                },
            })

print(f"Built {len(runs_kd_vs_ctrl)} (KD, control) UMAP runs across "
      f"{len(PLATES_TO_RUN)} plate(s); KD slice: {KD_SLICE} → {slice_tag}.")

submit_runs(
    runs_kd_vs_ctrl, memory=10000,
    dir=f"./{OUTDIR_NAME}/funova_screen_umap0_kd_vs_ctrl",
    submit=True, nova_home=NOVA_HOME, model_path=MODEL_PATH,
);

Built 15 (KD, control) UMAP runs across 1 plate(s); KD slice: slice(3, 6, None) → 3_6.
Data: +0 new, 15 already present (file now has 186 classes)
Plot: +0 new, 15 already present (file now has 186 classes)
  data classes kept (not regenerated): ['FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00004_00017_p3', 'FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00010_00031_p3', 'FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00035_00050_p3', 'FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00053_00059_p3', 'FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00111_00121_p3', 'FuNOVA_Screen_Data_plate3_NADSYN1_vs_non_targeting_00004_00017_p3', 'FuNOVA_Screen_Data_plate3_NADSYN1_vs_non_targeting_00010_00031_p3', 'FuNOVA_Screen_Data_plate3_NADSYN1_vs_non_targeting_00035_00050_p3', 'FuNOVA_Screen_Data_plate3_NADSYN1_vs_non_targeting_00053_00059_p3', 'FuNOVA_Screen_Data_plate3_NADSYN1_vs_non_targeting_00111_00121_p3', 'FuNOVA_Screen_Data_plate3_POLI_vs_non_targeting_00004_00017_p3', 'FuNOVA_Scre

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===



Job <139656> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_kd_vs_ctrl/FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00035_00050_p3_2.out -J umap_2 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00035_00050_p3 ./NOVA/manuscript/manuscript_plot_config_FuNOVA_Screen_generated/FuNOVA_Screen_Plot_plate3_MYCN_vs_non_targeting_00035_00050_p3_condition
Job <139657> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_kd_vs_ctrl/FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00053_00059_p3_3.out -J umap_3 python /home/projects/hornsteinlab/Colla

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===



Job <139658> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_kd_vs_ctrl/FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00111_00121_p3_4.out -J umap_4 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_plate3_MYCN_vs_non_targeting_00111_00121_p3 ./NOVA/manuscript/manuscript_plot_config_FuNOVA_Screen_generated/FuNOVA_Screen_Plot_plate3_MYCN_vs_non_targeting_00111_00121_p3_condition
Job <139659> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_kd_vs_ctrl/FuNOVA_Screen_Data_plate3_NADSYN1_vs_non_targeting_00004_00017_p3_5.out -J umap_5 python /home/projects/hornsteinlab/Co

...

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===



Job <139666> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_kd_vs_ctrl/FuNOVA_Screen_Data_plate3_POLI_vs_non_targeting_00035_00050_p3_12.out -J umap_12 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_plate3_POLI_vs_non_targeting_00035_00050_p3 ./NOVA/manuscript/manuscript_plot_config_FuNOVA_Screen_generated/FuNOVA_Screen_Plot_plate3_POLI_vs_non_targeting_00035_00050_p3_condition
Job <139667> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_kd_vs_ctrl/FuNOVA_Screen_Data_plate3_POLI_vs_non_targeting_00053_00059_p3_13.out -J umap_13 python /home/projects/hornsteinlab/C

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===

Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===



Job <139668> is submitted to queue <short>.

>>> submitting: bsub -q short -R rusage[mem=10000] -o ./Funova_Screen_UMAPS_logs/funova_screen_umap0_kd_vs_ctrl/FuNOVA_Screen_Data_plate3_POLI_vs_non_targeting_00111_00121_p3_14.out -J umap_14 python /home/projects/hornsteinlab/Collaboration/NOVA/runnables/generate_umaps_and_plot.py /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen ./NOVA/manuscript/manuscript_figures_data_config_FuNOVA_Screen_generated/FuNOVA_Screen_Data_plate3_POLI_vs_non_targeting_00111_00121_p3 ./NOVA/manuscript/manuscript_plot_config_FuNOVA_Screen_generated/FuNOVA_Screen_Plot_plate3_POLI_vs_non_targeting_00111_00121_p3_condition
Job <139669> is submitted to queue <short>.


Memory reservation is (MB): 10000
Memory Limit is (MB): 10000

===Your total amount of memory reservation for this job is (MB): 10000 ===



## Your runs go here

Add cells below with your own spec lists, copy patterns from the examples above.
Each `submit_runs(...)` call **overwrites** the two `_generated.py` files,
so submit one batch of runs at a time, then submit the bsubs before regenerating.

In [12]:
# Template — edit and run.
my_runs = [
    # {"data": {"name": "MyRun_1", "conditions": plate1_conditions[:10]},
    #  "plot": {"color_by": "condition", "umap_type": 0}},
]
# submit_runs(my_runs, memory=10000, dir="my_runs", submit=False);